<a href="https://colab.research.google.com/github/dzudo/try_git/blob/master/custom_microwakeword_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# microWakeWord — Custom wake word training

Full pipeline: test pronunciations, generate samples, train, export `.tflite`.

## Workarounds applied
- `datasets==2.21.0` (newer versions require torchcodec which breaks audio decoding)
- `piper-tts` needed for piper-sample-generator
- Augmentation data downloaded via soundfile/pydub (bypasses datasets Audio decoder)
- microWakeWord `utils.py` patched: stride assertion replaced with truncation
- `clip_duration_ms` must produce frames divisible by stride (3). Use 1500 or 1800, NOT 2000
- `dinner_party_eval` dataset removed from training config (causes OOM on Colab free tier)

In [1]:
# =============================================================================
# Cell 1: CONFIGURATION — Edit these parameters before running
# =============================================================================
# IMPORTANT: Runtime → Change runtime type → T4 GPU (before running any cell)
# Training works on CPU too but takes ~2-3h instead of ~20min.
# If GPU is unavailable (free tier quota), wait 12-24h for reset or use CPU.
# =============================================================================

import json

CONFIG = {
    'WAKE_WORD_NAME': 'ej_bogdan',
    'CLIP_DURATION_MS': 1800,              # fraza dwuwyrazowa (1500 lub 1800, nie 2000)
    'PROBABILITY_CUTOFF': 0.85,

    # Warianty wymowy do odsłuchu w komórce 5; w 5b wybierz te, które brzmią naturalnie
    'TEST_VARIANTS': ['ej bogdan', 'ej, bogdan', 'ejj bogdan', 'ej bogdan!', 'ej... bogdan'],
    'SAMPLES_PER_COMBO': 400,              # więcej niż domyślne 250 - polskich głosów jest mało

    'LANGUAGE': 'pl_PL',
    'VOICE_QUALITY': 'medium',

    'NEGATIVE_CLASS_WEIGHT': 20,
    'TRAINING_STEPS': 10000,
    'EVAL_STEP_INTERVAL': 2000,

    'USE_DINNER_PARTY_EVAL': False,
}

with open('/content/config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

print(f"Wake word: {CONFIG['WAKE_WORD_NAME']}")
print(f"Clip duration: {CONFIG['CLIP_DURATION_MS']}ms")
print(f"Test variants: {CONFIG['TEST_VARIANTS']}")
print(f"Language: {CONFIG['LANGUAGE']}, quality: {CONFIG['VOICE_QUALITY']}")
print(f"Dinner party eval: {'ON' if CONFIG['USE_DINNER_PARTY_EVAL'] else 'OFF'}")
print('Config saved to /content/config.json')

Wake word: ej_bogdan
Clip duration: 1800ms
Test variants: ['ej bogdan', 'ej, bogdan', 'ejj bogdan', 'ej bogdan!', 'ej... bogdan']
Language: pl_PL, quality: medium
Dinner party eval: OFF
Config saved to /content/config.json


In [2]:
# Cell 2: Install dependencies
# Safe to re-run — skips what's already installed/cloned.

%pip install 'datasets==2.21.0'
%pip install 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'
!test -d /content/microWakeWord || git clone https://github.com/kahrendt/microWakeWord /content/microWakeWord
%pip install -e /content/microWakeWord
%pip install piper-tts soundfile pydub

import importlib, site
importlib.invalidate_caches()
site.main()

import datasets
try:
    import microwakeword
    print(f'microwakeword OK')
except ModuleNotFoundError:
    raise RuntimeError('microWakeWord install failed. Scroll up for pip errors.')

if datasets.__version__ != '2.21.0':
    print('\n*** RESTART REQUIRED: old datasets version cached in memory. ***')
    print('*** Runtime → Restart session, then continue from Cell 3. ***')
else:
    print('\nReady. Continue with Cell 3.')

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 7.9 MB/s  0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.12.0
    Uninstalling fsspec-2025.12.0:
      Successfully uninstalled fsspec-2025.12.0
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Successfully uninstalled dill-0.4.1
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.19
    Uninstalling multiprocess-0.70.19:
      Successfully uninstalled multiprocess-0.70.19
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [datasets]
ERROR: pip's dependency resolver does not currently take into acc

In [3]:
# Cell 3: Patch microWakeWord stride assertion bug
# Verifies session was restarted after Cell 2.

import importlib
try:
    import datasets
    importlib.reload(datasets)
    assert datasets.__version__ == '2.21.0', \
        f'datasets {datasets.__version__} loaded, need 2.21.0. Restart session (Runtime → Restart session), then run Cell 1 and continue from here.'
except ImportError:
    raise RuntimeError('datasets not installed. Run Cell 2 first.')

with open('/content/microWakeWord/microwakeword/utils.py', 'r') as f:
    code = f.read()

old = '            assert spectrogram.shape[0] % stride == 0'
if old in code:
    new = '''            remainder = spectrogram.shape[0] % stride
            if remainder != 0:
                spectrogram = spectrogram[:spectrogram.shape[0] - remainder]'''
    code = code.replace(old, new)
    with open('/content/microWakeWord/microwakeword/utils.py', 'w') as f:
        f.write(code)
    print('Patched')
else:
    print('Already patched')

Patched


In [19]:
# Cell 4: Download piper-sample-generator + voices (auto-detected from LANGUAGE)

import os, json, urllib.request

with open('/content/config.json') as f:
    CONFIG = json.load(f)

if not os.path.exists('./piper-sample-generator'):
    !git clone https://github.com/rhasspy/piper-sample-generator
    !pip install torch torchaudio piper-phonemize-cross==1.2.1

voices_json_path = '/content/voices.json'
if not os.path.exists(voices_json_path):
    urllib.request.urlretrieve(
        'https://huggingface.co/rhasspy/piper-voices/resolve/main/voices.json',
        voices_json_path
    )

with open(voices_json_path) as f:
    all_voices = json.load(f)

VOICES = {}
lang = CONFIG['LANGUAGE']
quality = CONFIG['VOICE_QUALITY']
lang_short = lang.split('_')[0]
for key, info in all_voices.items():
    if not key.startswith(lang):
        continue
    if info.get('quality') != quality:
        continue
    onnx_file = f'{key}.onnx'
    parts = key.replace(f'{lang}-', '').rsplit('-', 1)
    speaker = parts[0]
    url = f'https://huggingface.co/rhasspy/piper-voices/resolve/main/{lang_short}/{lang}/{speaker}/{quality}/{onnx_file}'
    VOICES[key] = url

print(f'Found {len(VOICES)} voices for {lang}/{quality}: {list(VOICES.keys())}')

for name, url in VOICES.items():
    onnx_path = f'/content/{name}.onnx'
    json_path = f'{onnx_path}.json'
    if not os.path.exists(onnx_path):
        !wget -q -O {onnx_path} '{url}'
        !wget -q -O {json_path} '{url}.json'

CONFIG['VOICES'] = VOICES
with open('/content/config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

print(f'Voices downloaded. Continue with Cell 5 to test variants.')

Found 3 voices for pl_PL/medium: ['pl_PL-darkman-medium', 'pl_PL-gosia-medium', 'pl_PL-mc_speech-medium']
Voices downloaded. Continue with Cell 5 to test variants.


In [5]:
# Cell 5: Test pronunciation variants (listen to all candidates)
# 1. Run this cell to hear each variant
# 2. Listen to the audio samples
# 3. In the next cell (5b), type the numbers of the ones you want to keep and press Enter

from IPython.display import Audio, display, HTML
import os, json

with open('/content/config.json') as f:
    CONFIG = json.load(f)

VOICES = CONFIG['VOICES']
MODEL = f'/content/{list(VOICES.keys())[0]}.onnx'

for i, word in enumerate(CONFIG['TEST_VARIANTS']):
    slug = word.replace(' ', '_')
    out_dir = f'/content/test_{slug}'
    os.makedirs(out_dir, exist_ok=True)

    !cd /content/piper-sample-generator && python -m piper_sample_generator \
        "{word}" \
        --model {MODEL} \
        --max-samples 1 \
        --output-dir {out_dir}

    display(HTML(f'<h3>#{i}: {word}</h3>'))
    display(Audio(f'{out_dir}/0.wav'))

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
DEBUG:__main__:Loading ['/content/pl_PL-darkman-medium.onnx']
DEBUG:piper.voice:Guessing voice config path: /content/pl_PL-darkman-medium.onnx.json
DEBUG:piper.voice:Using CUDA
/usr/local/lib/python3.13/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:197: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
INFO:__main__:Successfully loaded model(s)
DEBUG:piper.voice:text=ej bogdan, phonemes=[['ˈ', 'ɛ', 'j', ' ', 'b', 'ˈ', 'ɔ', 'ɡ', 'd', 'a', 'n']]


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
DEBUG:__main__:Loading ['/content/pl_PL-darkman-medium.onnx']
DEBUG:piper.voice:Guessing voice config path: /content/pl_PL-darkman-medium.onnx.json
DEBUG:piper.voice:Using CUDA
/usr/local/lib/python3.13/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:197: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
INFO:__main__:Successfully loaded model(s)
DEBUG:piper.voice:text=ej, bogdan, phonemes=[['ˈ', 'ɛ', 'j', ',', ' ', 'b', 'ˈ', 'ɔ', 'ɡ', 'd', 'a', 'n']]


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
DEBUG:__main__:Loading ['/content/pl_PL-darkman-medium.onnx']
DEBUG:piper.voice:Guessing voice config path: /content/pl_PL-darkman-medium.onnx.json
DEBUG:piper.voice:Using CUDA
/usr/local/lib/python3.13/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:197: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
INFO:__main__:Successfully loaded model(s)
DEBUG:piper.voice:text=ejj bogdan, phonemes=[['ˈ', 'ɛ', 'j', 'j', ' ', 'b', 'ˈ', 'ɔ', 'ɡ', 'd', 'a', 'n']]


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
DEBUG:__main__:Loading ['/content/pl_PL-darkman-medium.onnx']
DEBUG:piper.voice:Guessing voice config path: /content/pl_PL-darkman-medium.onnx.json
DEBUG:piper.voice:Using CUDA
/usr/local/lib/python3.13/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:197: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
INFO:__main__:Successfully loaded model(s)
DEBUG:piper.voice:text=ej bogdan!, phonemes=[['ˈ', 'ɛ', 'j', ' ', 'b', 'ˈ', 'ɔ', 'ɡ', 'd', 'a', 'n', '!']]


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
DEBUG:__main__:Loading ['/content/pl_PL-darkman-medium.onnx']
DEBUG:piper.voice:Guessing voice config path: /content/pl_PL-darkman-medium.onnx.json
DEBUG:piper.voice:Using CUDA
/usr/local/lib/python3.13/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:197: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
INFO:__main__:Successfully loaded model(s)
DEBUG:piper.voice:text=ej... bogdan, phonemes=[['ˈ', 'ɛ', 'j', 'b', 'ˈ', 'ɔ', 'ɡ', 'd', 'a', 'n']]


In [6]:
# Cell 5b: Select which variants to keep for training
# 1. Run this cell — it will show the variant list and a text input box
# 2. Type the variant numbers to keep, separated by commas (e.g. 0,1)
# 3. Press Enter

import json

with open('/content/config.json') as f:
    CONFIG = json.load(f)

print('Available variants:')
for i, word in enumerate(CONFIG['TEST_VARIANTS']):
    print(f'  #{i}: {word}')

selection = input('\nEnter numbers to keep (comma-separated): ')
indices = [int(x.strip()) for x in selection.split(',')]
CONFIG['VARIANTS'] = [CONFIG['TEST_VARIANTS'][i] for i in indices]

with open('/content/config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

total = len(CONFIG['VARIANTS']) * len(CONFIG['VOICES']) * CONFIG['SAMPLES_PER_COMBO']
print(f"\nSelected: {CONFIG['VARIANTS']}")
print(f'Samples: {len(CONFIG["VARIANTS"])} variants x {len(CONFIG["VOICES"])} voices x {CONFIG["SAMPLES_PER_COMBO"]} = {total} total')

Available variants:
  #0: ej bogdan
  #1: ej, bogdan
  #2: ejj bogdan
  #3: ej bogdan!
  #4: ej... bogdan

Enter numbers to keep (comma-separated): 0,1,2,3,4

Selected: ['ej bogdan', 'ej, bogdan', 'ejj bogdan', 'ej bogdan!', 'ej... bogdan']
Samples: 5 variants x 3 voices x 400 = 6000 total


In [7]:
# Cell 6: Generate training samples

import os, json

with open('/content/config.json') as f:
    CONFIG = json.load(f)

VOICES = CONFIG['VOICES']
os.makedirs('/content/generated_samples', exist_ok=True)

MODELS = [f'/content/{name}.onnx' for name in VOICES.keys()]

sample_idx = 0
for word in CONFIG['VARIANTS']:
    for model in MODELS:
        voice_name = model.split('/')[-1].split('-')[2]
        tmp_dir = f'/content/tmp_gen_{word.replace(" ","_")}_{voice_name}'

        !cd /content/piper-sample-generator && python -m piper_sample_generator \
            "{word}" \
            --model {model} \
            --max-samples {CONFIG['SAMPLES_PER_COMBO']} \
            --batch-size 50 \
            --output-dir {tmp_dir}

        for f in sorted(os.listdir(tmp_dir)):
            if f.endswith('.wav'):
                os.rename(f'{tmp_dir}/{f}', f'/content/generated_samples/{sample_idx}.wav')
                sample_idx += 1

print(f'Total samples: {sample_idx}')

Strumieniowane dane wyjściowe obcięte do 5000 ostatnich wierszy.
DEBUG:piper.voice:text=ej bogdan, phonemes=[['ˈ', 'ɛ', 'j', ' ', 'b', 'ˈ', 'ɔ', 'ɡ', 'd', 'a', 'n']]
DEBUG:piper.voice:text=ej bogdan, phonemes=[['ˈ', 'ɛ', 'j', ' ', 'b', 'ˈ', 'ɔ', 'ɡ', 'd', 'a', 'n']]
DEBUG:piper.voice:text=ej bogdan, phonemes=[['ˈ', 'ɛ', 'j', ' ', 'b', 'ˈ', 'ɔ', 'ɡ', 'd', 'a', 'n']]
DEBUG:piper.voice:text=ej bogdan, phonemes=[['ˈ', 'ɛ', 'j', ' ', 'b', 'ˈ', 'ɔ', 'ɡ', 'd', 'a', 'n']]
DEBUG:piper.voice:text=ej bogdan, phonemes=[['ˈ', 'ɛ', 'j', ' ', 'b', 'ˈ', 'ɔ', 'ɡ', 'd', 'a', 'n']]
DEBUG:piper.voice:text=ej bogdan, phonemes=[['ˈ', 'ɛ', 'j', ' ', 'b', 'ˈ', 'ɔ', 'ɡ', 'd', 'a', 'n']]
DEBUG:piper.voice:text=ej bogdan, phonemes=[['ˈ', 'ɛ', 'j', ' ', 'b', 'ˈ', 'ɔ', 'ɡ', 'd', 'a', 'n']]
DEBUG:piper.voice:text=ej bogdan, phonemes=[['ˈ', 'ɛ', 'j', ' ', 'b', 'ˈ', 'ɔ', 'ɡ', 'd', 'a', 'n']]
DEBUG:piper.voice:text=ej bogdan, phonemes=[['ˈ', 'ɛ', 'j', ' ', 'b', 'ˈ', 'ɔ', 'ɡ', 'd', 'a', 'n']]
DEBUG:piper.voice:text=ej 

In [ ]:
# Cell 6N: Polskie próbki negatywne (uruchom po komórce 6)
# Generuje polskie frazy, które NIE są słowem wybudzającym, tymi samymi głosami Piper.

import os, json, shutil

with open('/content/config.json') as f:
    CONFIG = json.load(f)

NEG_PHRASES = [
    # --- fragmenty frazy i podobne brzmienia (model musi wymagać całości) ---
    'ej', 'hej', 'ej, ty', 'ej, słuchaj', 'ej, chodź tu', 'ej, zobacz', 'ej, co jest',
    'ej, poczekaj', 'ej, no weź', 'hej, cześć', 'hej, co słychać',
    'Bogdan', 'Bogdanie', 'pan Bogdan', 'wujek Bogdan', 'Bogdan Nowak',
    'Bogdan przyszedł', 'Bogdan dzwonił, oddzwoń do niego', 'Hej, gdzie jest Bogdan?',
    'Bogna', 'Bogusław', 'Bodzio', 'bo dam', 'podam', 'Bóg dał', 'bogaty',
    'ej, Damian', 'ej, Marian', 'ej, Bartek', 'ej, Bożena', 'ej, Bogna', 'ej, Gosia',
    # --- fałszywe wykrycie z testów i frazy o podobnym rytmie ---
    'czego chcesz', 'czego chcesz?!', 'czego?', 'co chcesz', 'czego szukasz',
    'czego tu chcesz', 'czy chcesz', 'chcesz herbaty?', 'co jest?', 'co się dzieje?',
    'jeszcze', 'przecież', 'cześć', 'dzięki', 'proszę', 'przepraszam',
    # --- krótkie zwroty codzienne ---
    'włącz światło', 'wyłącz telewizor', 'która godzina?', 'jaka jest pogoda?',
    'dzień dobry', 'dobranoc', 'smacznego', 'na zdrowie', 'nie wiem', 'zobaczymy',
    'chodź na obiad', 'gdzie są klucze?', 'zamknij drzwi', 'otwórz okno', 'idę spać',
    'jutro rano', 'do zobaczenia', 'co na obiad?', 'nie teraz', 'zaraz wracam',
    'daj spokój', 'no dobra', 'jasne', 'oczywiście', 'w porządku', 'tak', 'nie',
    'może później', 'podaj mi pilota', 'gdzie jest kot?', 'pies chce wyjść',
    'wynieś śmieci', 'kupiłem chleb', 'zadzwoń do mamy', 'mama dzwoniła', 'jedziemy',
    'ubierz się ciepło', 'zimno tu', 'zrób kawę', 'poproszę wodę', 'gdzie jest telefon?',
    'naładuj telefon', 'mecz się zaczyna', 'obiad gotowy', 'nastaw pranie',
    'włącz zmywarkę', 'odkurz pokój', 'szybko, spóźnimy się', 'co tam?', 'no co ty',
    'serio?', 'chwileczkę', 'uwaga', 'zostaw to', 'gdzie idziesz?', 'chodź tutaj',
    # --- dłuższe zdania (mowa ciągła, telewizja) ---
    'Wczoraj byliśmy w kinie na nowym filmie.',
    'Nie zapomnij kupić mleka w drodze do domu.',
    'Dzisiaj w nocy temperatura spadnie poniżej zera.',
    'Prognoza pogody na najbliższy weekend.',
    'Proszę zamknąć drzwi od garażu.',
    'Czy możesz mi pomóc w kuchni?',
    'Ile to kosztuje?',
    'Muszę jeszcze odpisać na kilka maili.',
    'Dzieci wróciły już ze szkoły.',
    'Zostało jeszcze trochę zupy.',
    'Przyjdę po ciebie o szóstej.',
    'Gdzie położyłeś moje okulary?',
    'Ten serial jest naprawdę dobry.',
    'Pociąg ma dwadzieścia minut spóźnienia.',
    'W sobotę jedziemy do babci.',
    'Witamy w wieczornym wydaniu wiadomości.',
    'Na drogach panują trudne warunki.',
    'Zapraszamy po przerwie na reklamy.',
    'Bramka! Co za strzał z rzutu wolnego!',
    'Sejm przyjął ustawę większością głosów.',
    'Czy ktoś widział moją niebieską kurtkę?',
    'Jutro mamy spotkanie o dziewiątej.',
    'Zamówiłem pizzę, będzie za pół godziny.',
    'Trzeba wymienić żarówkę w łazience.',
    'Słyszałeś, co się wczoraj stało?',
    'Chcesz jeszcze kawałek ciasta?',
    'Ej, nie rób tego, bo się przewrócisz.',
    'Hej, dawno się nie widzieliśmy.',
]

NEG_DIR = '/content/polish_negatives'
NEG_PER_VOICE = 1200          # łącznie: liczba głosów x 1200

shutil.rmtree(NEG_DIR, ignore_errors=True)
os.makedirs(NEG_DIR, exist_ok=True)
with open('/content/polish_negatives.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(NEG_PHRASES))

MODELS = [f'/content/{name}.onnx' for name in CONFIG['VOICES'].keys()]
idx = 0
for model in MODELS:
    vname = os.path.basename(model)[:-5]
    tmp_dir = f'/content/tmp_neg_{vname}'
    shutil.rmtree(tmp_dir, ignore_errors=True)
    print(f'Generuję negatywy głosem {vname} ...')
    !cd /content/piper-sample-generator && python -m piper_sample_generator \
        /content/polish_negatives.txt \
        --model {model} \
        --max-samples {NEG_PER_VOICE} \
        --output-dir {tmp_dir}
    for fn in sorted(os.listdir(tmp_dir)):
        if fn.endswith('.wav'):
            os.rename(f'{tmp_dir}/{fn}', f'{NEG_DIR}/neg_{idx}.wav')
            idx += 1

print(f'Polskie negatywy: {idx} plików, {len(NEG_PHRASES)} fraz')
if idx == 0:
    raise RuntimeError('Nie wygenerowano żadnych negatywów - sprawdź wyjście powyżej')

Generuję negatywy głosem pl_PL-darkman-medium ...
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
DEBUG:__main__:Loading ['/content/pl_PL-darkman-medium.onnx']
DEBUG:piper.voice:Guessing voice config path: /content/pl_PL-darkman-medium.onnx.json
DEBUG:piper.voice:Using CUDA
/usr/local/lib/python3.13/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:197: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
INFO:__main__:Successfully loaded model(s)
DEBUG:piper.voice:text=ej, phonemes=[['ˈ', 'ɛ', 'j']]
DEBUG:piper.voice:text=hej, phonemes=[['x', 'ˈ', 'ɛ', 'j']]
DEBUG:piper.voice:text=ej, ty, phonemes=[['ˈ', 'ɛ', 'j', ',', ' ', 't', 'ˈ', 'ɨ']]
DEBUG:piper.voice:text=ej, słuchaj, phonemes=[['ˈ', 'ɛ', 'j', ',', ' ', 's

In [8]:
# Cell 7: Download augmentation data (MIT RIRs, AudioSet, FMA)

import soundfile as sf
import scipy
import os
import numpy as np
from pathlib import Path
from tqdm import tqdm
from datasets import load_dataset, Audio
import io

# MIT Room Impulse Responses
output_dir = './mit_rirs'
if not os.path.exists(output_dir) or len(os.listdir(output_dir)) == 0:
    os.makedirs(output_dir, exist_ok=True)
    rir_dataset = load_dataset('davidscripka/MIT_environmental_impulse_responses', split='train')
    rir_dataset = rir_dataset.cast_column('audio', Audio(decode=False))
    for i, row in enumerate(tqdm(rir_dataset)):
        audio_path = row['audio']['path']
        if audio_path and os.path.exists(audio_path):
            data, sr = sf.read(audio_path)
            if len(data.shape) > 1:
                data = data[:, 0]
            if sr != 16000:
                import librosa
                data = librosa.resample(data, orig_sr=sr, target_sr=16000)
            scipy.io.wavfile.write(os.path.join(output_dir, f'rir_{i}.wav'), 16000, (data*32767).astype(np.int16))

# AudioSet
if not os.path.exists('audioset'):
    os.mkdir('audioset')
    fname = 'bal_train09.tar'
    link = 'https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/' + fname
    !wget -O audioset/{fname} {link}
    !cd audioset && tar -xf bal_train09.tar

output_dir = './audioset_16k'
if not os.path.exists(output_dir) or len(os.listdir(output_dir)) == 0:
    os.makedirs(output_dir, exist_ok=True)
    for flac in tqdm(list(Path('audioset/audio').glob('**/*.flac'))):
        try:
            data, sr = sf.read(str(flac))
            if len(data.shape) > 1:
                data = data[:, 0]
            if sr != 16000:
                import librosa
                data = librosa.resample(data, orig_sr=sr, target_sr=16000)
            scipy.io.wavfile.write(os.path.join(output_dir, flac.stem + '.wav'), 16000, (data*32767).astype(np.int16))
        except Exception:
            pass

# FMA
if not os.path.exists('fma') or not os.path.exists('fma/fma_small'):
    os.makedirs('fma', exist_ok=True)
    fname = 'fma_xs.zip'
    link = 'https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/' + fname
    !wget -O fma/{fname} {link}
    !cd fma && unzip -q {fname}

output_dir = './fma_16k'
if not os.path.exists(output_dir) or len(os.listdir(output_dir)) == 0:
    os.makedirs(output_dir, exist_ok=True)
    from pydub import AudioSegment
    for mp3 in tqdm(list(Path('fma/fma_small').glob('**/*.mp3'))):
        try:
            audio = AudioSegment.from_mp3(str(mp3)).set_frame_rate(16000).set_channels(1)
            samples = np.array(audio.get_array_of_samples(), dtype=np.int16)
            scipy.io.wavfile.write(os.path.join(output_dir, mp3.stem + '.wav'), 16000, samples)
        except Exception:
            pass

print('Augmentation data ready')

Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/270 [00:00<?, ? examples/s]

100%|██████████| 270/270 [00:00<00:00, 1951.01it/s]


--2026-11-23 09:39:28--  https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar
Resolving huggingface.co (huggingface.co)... 18.164.174.23, 18.164.174.118, 18.164.174.17, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.23|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-11-23 09:39:28 ERROR 404: Not Found.

tar: This does not look like a tar archive
tar: Exiting with failure status due to previous errors


0it [00:00, ?it/s]

--2026-11-23 09:39:28--  https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip
Resolving huggingface.co (huggingface.co)... 18.164.174.23, 18.164.174.118, 18.164.174.17, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.23|:443... connected.
HTTP request sent, awaiting response... 

302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/66ca2060627175be3322dcd0/b15b33649259980016c036a0b2a0fee516e0a9acf3f1adcc561e0abd5efde02b?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27fma_xs.zip%3B+filename%3D%22fma_xs.zip%22%3B&response-content-type=application%2Fzip&X-Xet-Cas-Uid=public&xip=pu7fVJdB4Kw&user_id=public&Expires=1795430368&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjZjYTIwNjA2MjcxNzViZTMzMjJkY2QwL2IxNWIzMzY0OTI1OTk4MDAxNmMwMzZhMGIyYTBmZWU1MTZlMGE5YWNmM2YxYWRjYzU2MWUwYWJkNWVmZGUwMmJcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomcmVzcG9uc2UtY29udGVudC10eXBlPSomWC1YZXQtQ2FzLVVpZD1wdWJsaWMmeGlwPXB1N2ZWSmRCNEt3JnVzZXJfaWQ9cHVibGljIiwiQ29uZGl0aW9uIjp7IkRhdGVMZXNzVGhhbiI6eyJFcG9jaFRpbWUiOjE3OTU0MzAzNjh9fX1dfQ__&Signature=MEQCIC9r%7EZgAPT2Y8YfMvYwIWILCGwQyXNY9vNEy8YNtplPsAiB%7E-BHXnbPjV-14sDJW949ujUuhlDPsV7Poyx31xpehLw__&Key-Pair-Id=01KXEF4KZ1B6FV465MAWR4M21F&Hash-Algorithm=SHA256 [following]
--2026-11

/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
100%|██████████| 210/210 [01:11<00:00,  2.95it/s]

Augmentation data ready


In [9]:
# Naprawa AudioSet (uruchom po komórce 7, przed komórką 8)
import os, shutil, subprocess
from pathlib import Path
import soundfile as sf, scipy.io.wavfile, numpy as np
from tqdm import tqdm

shutil.rmtree('audioset', ignore_errors=True)
shutil.rmtree('audioset_16k', ignore_errors=True)
os.makedirs('audioset', exist_ok=True)

urls = [
    'https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar',
    'https://huggingface.co/datasets/agkphysics/AudioSet/resolve/c2f9c5b3e3211790680f2890608af7dc73ddc540/data/bal_train09.tar',
    'https://huggingface.co/datasets/agkphysics/AudioSet/resolve/c051cda6f8c73b556e691adcff7aa3070ca58456/bal_train09.tar',
]
tar_path = 'audioset/bal_train09.tar'
ok = False
for u in urls:
    print('Próbuję:', u)
    r = subprocess.run(['wget', '-q', '-O', tar_path, u])
    if r.returncode == 0 and os.path.getsize(tar_path) > 100_000_000:
        ok = True
        print('  pobrano', os.path.getsize(tar_path) // 1_000_000, 'MB')
        break
    print('  nieudane')
if not ok:
    raise RuntimeError('AudioSet niedostępny pod żadnym adresem - patrz wariant awaryjny')

subprocess.run(['tar', '-xf', 'bal_train09.tar'], cwd='audioset', check=True)

os.makedirs('audioset_16k', exist_ok=True)
flacs = list(Path('audioset').glob('**/*.flac'))
print('Plików FLAC:', len(flacs))
for flac in tqdm(flacs):
    try:
        data, sr = sf.read(str(flac))
        if len(data.shape) > 1:
            data = data[:, 0]
        if sr != 16000:
            import librosa
            data = librosa.resample(data, orig_sr=sr, target_sr=16000)
        scipy.io.wavfile.write(os.path.join('audioset_16k', flac.stem + '.wav'), 16000,
                               (data * 32767).astype(np.int16))
    except Exception:
        pass
print('audioset_16k:', len(os.listdir('audioset_16k')), 'plików')

Próbuję: https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar
  nieudane
Próbuję: https://huggingface.co/datasets/agkphysics/AudioSet/resolve/c2f9c5b3e3211790680f2890608af7dc73ddc540/data/bal_train09.tar
  pobrano 968 MB
Plików FLAC: 685


100%|██████████| 685/685 [00:35<00:00, 19.25it/s]

audioset_16k: 685 plików


In [10]:
# Cell 8: Generate spectrogram features

try:
    from microwakeword.audio.augmentation import Augmentation
    from microwakeword.audio.clips import Clips
    from microwakeword.audio.spectrograms import SpectrogramGeneration
except ModuleNotFoundError:
    raise RuntimeError('microWakeWord not installed. Re-run from Cell 2 (VM was reset).')

from mmap_ninja.ragged import RaggedMmap
import os

clips = Clips(
    input_directory='generated_samples',
    file_pattern='*.wav',
    max_clip_duration_s=None,
    remove_silence=False,
    random_split_seed=10,
    split_count=0.1,
)

augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.1,
        'TanhDistortion': 0.1,
        'PitchShift': 0.1,
        'BandStopFilter': 0.1,
        'AddColorNoise': 0.1,
        'AddBackgroundNoise': 0.75,
        'Gain': 1.0,
        'RIR': 0.5,
    },
    impulse_paths=['mit_rirs'],
    background_paths=['fma_16k', 'audioset_16k'],
    background_min_snr_db=-5,
    background_max_snr_db=10,
    min_jitter_s=0.195,
    max_jitter_s=0.205,
)

output_dir = 'generated_augmented_features'
os.makedirs(output_dir, exist_ok=True)

for split in ['training', 'validation', 'testing']:
    out_dir = os.path.join(output_dir, split)
    os.makedirs(out_dir, exist_ok=True)

    split_name = {'training': 'train', 'validation': 'validation', 'testing': 'test'}[split]
    repetition = 2 if split == 'training' else 1
    slide = 1 if split == 'testing' else 10

    spectrograms = SpectrogramGeneration(
        clips=clips,
        augmenter=augmenter,
        slide_frames=slide,
        step_ms=10,
    )

    RaggedMmap.from_generator(
        out_dir=os.path.join(out_dir, 'wakeword_mmap'),
        sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
        batch_size=100,
        verbose=True,
    )

print('Spectrogram features ready')

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

Spectrogram features ready


In [ ]:
# Cell 8N: Cechy (spektrogramy) dla polskich negatywów (uruchom po komórce 8)

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from mmap_ninja.ragged import RaggedMmap
import os, shutil

neg_clips = Clips(
    input_directory='/content/polish_negatives',
    file_pattern='*.wav',
    max_clip_duration_s=None,
    remove_silence=False,
    random_split_seed=10,
    split_count=0.1,
)

# Te same parametry augmentacji co dla próbek pozytywnych (komórka 8);
# używa tylko tych katalogów tła, które faktycznie zawierają pliki.
bg_paths = [p for p in ['fma_16k', 'audioset_16k'] if os.path.isdir(p) and len(os.listdir(p)) > 0]
print('Tło do augmentacji:', bg_paths)

neg_augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.1,
        'TanhDistortion': 0.1,
        'PitchShift': 0.1,
        'BandStopFilter': 0.1,
        'AddColorNoise': 0.1,
        'AddBackgroundNoise': 0.75,
        'Gain': 1.0,
        'RIR': 0.5,
    },
    impulse_paths=['mit_rirs'],
    background_paths=bg_paths,
    background_min_snr_db=-5,
    background_max_snr_db=10,
    min_jitter_s=0.195,
    max_jitter_s=0.205,
)

output_dir = 'polish_negative_features'
shutil.rmtree(output_dir, ignore_errors=True)

for split in ['training', 'validation', 'testing']:
    out_dir = os.path.join(output_dir, split)
    os.makedirs(out_dir, exist_ok=True)

    split_name = {'training': 'train', 'validation': 'validation', 'testing': 'test'}[split]
    repetition = 2 if split == 'training' else 1
    slide = 1 if split == 'testing' else 10

    spectrograms = SpectrogramGeneration(
        clips=neg_clips,
        augmenter=neg_augmenter,
        slide_frames=slide,
        step_ms=10,
    )

    mmap_path = os.path.join(out_dir, 'polish_negatives_mmap')
    RaggedMmap.from_generator(
        out_dir=mmap_path,
        sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
        batch_size=100,
        verbose=True,
    )
    print(f'{split}: {len(RaggedMmap(mmap_path))} spektrogramów')

print('Polish negative features ready')

In [11]:
# Cell 9: Download pre-generated negative datasets

import os

output_dir = './negative_datasets'
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    link_root = 'https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/'
    filenames = ['dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip']
    for fname in filenames:
        link = link_root + fname
        zip_path = f'negative_datasets/{fname}'
        !wget -O {zip_path} {link}
        !unzip -q {zip_path} -d {output_dir}

print('Negative datasets ready')

--2026-11-23 09:50:29--  https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/dinner_party.zip
Resolving huggingface.co (huggingface.co)... 18.164.174.118, 18.164.174.23, 18.164.174.55, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.118|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/65e327bc1445a768ed343b8c/228d7e72cd5fdc4e6e57da36b88a4c227d34cb8dc44041078b4c4b65dc75848d?xip=pu7fVJdB4Kw&response-content-type=application%2Fzip&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27dinner_party.zip%3B+filename%3D%22dinner_party.zip%22%3B&user_id=public&X-Xet-Cas-Uid=public&Expires=1795431029&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjVlMzI3YmMxNDQ1YTc2OGVkMzQzYjhjLzIyOGQ3ZTcyY2Q1ZmRjNGU2ZTU3ZGEzNmI4OGE0YzIyN2QzNGNiOGRjNDQwNDEwNzhiNGM0YjY1ZGM3NTg0OGRcXD94aXA9cHU3ZlZKZEI0S3cmcmVzcG9uc2UtY29udGVudC10eXBlPSomcmVzcG9uc2UtY29udGVudC1ka

In [12]:
# Kontrola i naprawa zbiorów negatywnych (uruchom po komórce 9, przed komórką 10)
import os, glob

root = 'negative_datasets'
link_root = 'https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/'
os.makedirs(root, exist_ok=True)

for name in ['speech', 'dinner_party', 'no_speech']:
    d = f'{root}/{name}'
    n = len(glob.glob(f'{d}/**/*', recursive=True)) if os.path.isdir(d) else 0
    print(f'{d}: {n} plików')
    if n == 0:
        z = f'{root}/{name}.zip'
        print(f'  pobieram {name}.zip ...')
        !wget -q --show-progress -O {z} {link_root}{name}.zip
        !unzip -q -o {z} -d {root}
        n = len(glob.glob(f'{d}/**/*', recursive=True))
        print(f'  po pobraniu: {n} plików')
        if n == 0:
            raise RuntimeError(f'{name}: zbiór nadal pusty - sprawdź wyjście wget/unzip powyżej')

print('\nStruktura katalogów:')
!find negative_datasets -maxdepth 3 -type d | sort | head -40
!du -sh negative_datasets/*
!df -h /content | tail -1

negative_datasets/speech: 169 plików
negative_datasets/dinner_party: 113 plików
negative_datasets/no_speech: 225 plików

Struktura katalogów:
negative_datasets
negative_datasets/dinner_party
negative_datasets/dinner_party_eval
negative_datasets/dinner_party_eval/testing_ambient
negative_datasets/dinner_party_eval/testing_ambient/dipco_u01_ch1_mmap
negative_datasets/dinner_party_eval/validation_ambient
negative_datasets/dinner_party_eval/validation_ambient/chime6_dev_eval_mmap
negative_datasets/dinner_party/training
negative_datasets/dinner_party/training/chime6_train_u01_ch1_mmap
negative_datasets/dinner_party/training/chime6_train_u02_ch1_mmap
negative_datasets/no_speech
negative_datasets/no_speech/training
negative_datasets/no_speech/training/fma_medium_mmap
negative_datasets/no_speech/training/fsd50k_no_speech_mmap
negative_datasets/no_speech/training/fsd50k_speech_mmap
negative_datasets/no_speech/training/wham_train_mmap
negative_datasets/speech
negative_datasets/speech/training
ne

In [13]:
# Cell 10: Training config

import json, yaml

with open('/content/config.json') as f:
    CONFIG = json.load(f)

config = {}
config['window_step_ms'] = 10
config['train_dir'] = f"trained_models/{CONFIG['WAKE_WORD_NAME']}"

config['features'] = [
    {
        'features_dir': 'generated_augmented_features',
        'sampling_weight': 2.0,
        'penalty_weight': 1.0,
        'truth': True,
        'truncation_strategy': 'truncate_start',
        'type': 'mmap',
    },
    {
        'features_dir': 'negative_datasets/speech',
        'sampling_weight': 10.0,
        'penalty_weight': 1.0,
        'truth': False,
        'truncation_strategy': 'random',
        'type': 'mmap',
    },
    {
        'features_dir': 'negative_datasets/dinner_party',
        'sampling_weight': 10.0,
        'penalty_weight': 1.0,
        'truth': False,
        'truncation_strategy': 'random',
        'type': 'mmap',
    },
    {
        'features_dir': 'negative_datasets/no_speech',
        'sampling_weight': 5.0,
        'penalty_weight': 1.0,
        'truth': False,
        'truncation_strategy': 'random',
        'type': 'mmap',
    },
]

if CONFIG.get('USE_DINNER_PARTY_EVAL'):
    config['features'].append({
        'features_dir': 'negative_datasets/dinner_party_eval',
        'sampling_weight': 1.0,
        'penalty_weight': 1.0,
        'truth': False,
        'truncation_strategy': 'split',
        'type': 'mmap',
    })

config['training_steps'] = [CONFIG['TRAINING_STEPS']]
config['positive_class_weight'] = [1]
config['negative_class_weight'] = [CONFIG['NEGATIVE_CLASS_WEIGHT']]
config['learning_rates'] = [0.001]
config['batch_size'] = 128
config['time_mask_max_size'] = [0]
config['time_mask_count'] = [0]
config['freq_mask_max_size'] = [0]
config['freq_mask_count'] = [0]
config['eval_step_interval'] = CONFIG['EVAL_STEP_INTERVAL']
config['clip_duration_ms'] = CONFIG['CLIP_DURATION_MS']
config['target_minimization'] = 0.9
config['minimization_metric'] = None
config['maximization_metric'] = 'average_viable_recall'

with open('training_parameters.yaml', 'w') as f:
    yaml.dump(config, f)

dp_status = ' + dinner_party_eval' if CONFIG.get('USE_DINNER_PARTY_EVAL') else ''
print(f"Training config saved for \"{CONFIG['WAKE_WORD_NAME']}\" ({CONFIG['CLIP_DURATION_MS']}ms, neg_weight={CONFIG['NEGATIVE_CLASS_WEIGHT']}{dp_status})")

Training config saved for "ej_bogdan" (1800ms, neg_weight=20)


In [ ]:
# Cell 10b: Dodanie polskich negatywów do konfiguracji treningu (uruchom po komórce 10)

import yaml

with open('training_parameters.yaml') as f:
    config = yaml.safe_load(f)

config['features'] = [x for x in config['features']
                      if x['features_dir'] != 'polish_negative_features']
config['features'].append({
    'features_dir': 'polish_negative_features',
    'sampling_weight': 5.0,
    'penalty_weight': 1.0,
    'truth': False,
    'truncation_strategy': 'truncate_start',   # fraza na końcu okna, jak w próbkach pozytywnych
    'type': 'mmap',
})

with open('training_parameters.yaml', 'w') as f:
    yaml.dump(config, f)

for x in config['features']:
    print(f"{x['features_dir']:32s} truth={x['truth']!s:5s} sampling_weight={x['sampling_weight']}")

In [17]:
# Cell 11: Train model
# ~20 min on T4 GPU. May appear stuck — Colab doesn't print mini-batch results.
!rm -rf trained_models/ej_bogdan
!python -m microwakeword.model_train_eval \
--training_config='training_parameters.yaml' \
--train 1 \
--restore_checkpoint 1 \
--test_tf_nonstreaming 0 \
--test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 \
--test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 \
--use_weights "last_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block  "1, 1, 1, 1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

INFO:absl:Loading and analyzing data sets.
W0000 00:00:1795431359.080613   33559 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (1, 214, 40)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims         │ (1, 214, 1, 40)   │          0 │ input_layer[0][0] │
│ (ExpandDims)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stream (Stream)     │ (1, 70, 1, 32)    │      6,

In [18]:
# Cell 12: Download trained model

from google.colab import files
import os

for root, dirs, fnames in os.walk('trained_models'):
    for f in fnames:
        if f.endswith('.tflite'):
            path = os.path.join(root, f)
            print(f'Downloading: {path}')
            files.download(path)

Downloading: trained_models/ej_bogdan/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>